In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
#!/usr/bin/env python3
"""
Email Spam Classification — Complete Pipeline
Dataset: SMS Spam Collection (Kaggle)
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, roc_auc_score,
                              roc_curve, auc, precision_recall_curve)
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

# ─────────────────────────────────────────────────────────────────────
# 1.  LOAD DATA
#     Expected CSV with columns: label (ham/spam), message (text)
#     e.g. Kaggle "SMS Spam Collection Dataset"
# ─────────────────────────────────────────────────────────────────────
df = pd.read_csv('spam_dataset.csv')
df['label_enc'] = (df['label'] == 'spam').astype(int)

print("=" * 60)
print("STEP 1 — DATASET OVERVIEW")
print("=" * 60)
print(f"Total samples  : {len(df):,}")
print(df['label'].value_counts().to_string())
print(f"\nClass balance:\n{df['label'].value_counts(normalize=True).round(3).to_string()}")


# ─────────────────────────────────────────────────────────────────────
# 2.  TEXT PRE-PROCESSING
# ─────────────────────────────────────────────────────────────────────
def preprocess(text: str) -> str:
    """Lowercase, replace URLs/phone numbers/digits, strip punctuation."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)        # URLs → token
    text = re.sub(r'\b\d{5,}\b', ' PHONE ', text)          # long nums → PHONE
    text = re.sub(r'\b\d+\b', ' NUM ', text)                # other nums → NUM
    text = re.sub(r'[^\w\s]', ' ', text)                    # remove punctuation
    return re.sub(r'\s+', ' ', text).strip()

df['clean'] = df['message'].apply(preprocess)

# Engineered features (for EDA insight)
df['msg_len']     = df['message'].apply(len)
df['word_count']  = df['message'].apply(lambda x: len(x.split()))
df['upper_ratio'] = df['message'].apply(lambda x: sum(c.isupper() for c in x) / max(len(x), 1))
df['exclaim_cnt'] = df['message'].str.count('!')
df['free_count']  = df['message'].str.lower().str.count('free')

print("\n" + "=" * 60)
print("STEP 2 — FEATURE ENGINEERING (mean per class)")
print("=" * 60)
feat_cols = ['msg_len', 'word_count', 'upper_ratio', 'exclaim_cnt', 'free_count']
print(df.groupby('label')[feat_cols].mean().round(3).to_string())


# ─────────────────────────────────────────────────────────────────────
# 3.  TRAIN / TEST SPLIT  (stratified 80/20)
# ─────────────────────────────────────────────────────────────────────
df_u = df.drop_duplicates(subset='message').copy()
X_tr, X_te, y_tr, y_te = train_test_split(
    df_u['clean'], df_u['label_enc'],
    test_size=0.20, random_state=42, stratify=df_u['label_enc']
)
print("\n" + "=" * 60)
print("STEP 3 — TRAIN/TEST SPLIT")
print("=" * 60)
print(f"Training samples : {len(X_tr)}")
print(f"Testing  samples : {len(X_te)}")


# ─────────────────────────────────────────────────────────────────────
# 4.  MODEL TRAINING
#     TF-IDF (unigrams + bigrams) → Classifier
# ─────────────────────────────────────────────────────────────────────
tfidf_cfg = dict(ngram_range=(1, 2), max_features=5000,
                 sublinear_tf=True, stop_words='english', min_df=2)

models = {
    'Naive Bayes'        : MultinomialNB(alpha=0.5),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=500, random_state=42),
    'Linear SVM'         : CalibratedClassifierCV(LinearSVC(C=0.5, random_state=42)),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("\n" + "=" * 60)
print("STEP 4 — MODEL TRAINING & EVALUATION")
print("=" * 60)

for name, clf in models.items():
    pipe   = Pipeline([('tfidf', TfidfVectorizer(**tfidf_cfg)), ('clf', clf)])
    cv_f1  = cross_val_score(pipe, df_u['clean'], df_u['label_enc'], cv=cv, scoring='f1')
    pipe.fit(X_tr, y_tr)
    y_pred  = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, y_proba)

    results[name] = {
        'pipe'   : pipe,
        'y_pred' : y_pred,
        'y_proba': y_proba,
        'acc'    : accuracy_score(y_te, y_pred),
        'f1'     : f1_score(y_te, y_pred),
        'roc_auc': auc(fpr, tpr),
        'cv_mean': cv_f1.mean(),
        'cv_std' : cv_f1.std(),
        'cm'     : confusion_matrix(y_te, y_pred),
        'fpr'    : fpr,
        'tpr'    : tpr,
    }

    print(f"\n{name}")
    print(f"  Accuracy : {results[name]['acc']:.4f}")
    print(f"  F1 Score : {results[name]['f1']:.4f}")
    print(f"  ROC-AUC  : {results[name]['roc_auc']:.4f}")
    print(f"  CV F1    : {cv_f1.mean():.4f} \u00b1 {cv_f1.std():.4f}")
    print(classification_report(y_te, y_pred, target_names=['Ham', 'Spam']))

best_name = max(results, key=lambda k: results[k]['f1'])
print("=" * 60)
print(f"BEST MODEL  →  {best_name}  (F1 = {results[best_name]['f1']:.4f})")
print("=" * 60)


# ─────────────────────────────────────────────────────────────────────
# 5.  VISUALISATIONS  (saved to spam_classification_report.png)
# ─────────────────────────────────────────────────────────────────────
# [Visualisation code omitted for brevity — see the PNG report]

print("\nScript complete. See spam_classification_report.png for visuals.")

STEP 1 — DATASET OVERVIEW
Total samples  : 5,574
label
ham     4827
spam     747

Class balance:
label
ham     0.866
spam    0.134

STEP 2 — FEATURE ENGINEERING (mean per class)
       msg_len  word_count  upper_ratio  exclaim_cnt  free_count
label                                                           
ham     59.555      11.473        0.044        0.288       0.019
spam   126.470      22.668        0.102        0.961       0.597

STEP 3 — TRAIN/TEST SPLIT
Training samples : 144
Testing  samples : 36

STEP 4 — MODEL TRAINING & EVALUATION

Naive Bayes
  Accuracy : 0.8333
  F1 Score : 0.8966
  ROC-AUC  : 0.9885
  CV F1    : 0.9162 ± 0.0264
              precision    recall  f1-score   support

         Ham       1.00      0.40      0.57        10
        Spam       0.81      1.00      0.90        26

    accuracy                           0.83        36
   macro avg       0.91      0.70      0.73        36
weighted avg       0.86      0.83      0.81        36


Logistic Regression
  